# Module 3 — Intent

**What you are doing today:** teaching your bot to work out the *goal* behind a
message, instead of hunting for one exact keyword.

It gets there by **counting**. No machine learning, no library, nothing
installed. Six word lists and a `for` loop.

| Part | What you do |
| --- | --- |
| **A — Follow along** | Run each cell, read what to notice. Nothing to write. |
| **B — Your turn** | Six tasks. Now you write the code. |

**Have your 10 collected messages open.** You need them in task B5.

Work in pairs. Swap who types between the two parts.


---
# Part A — Follow along

Seven short steps, A1 to A7, after one setup cell. **Run each cell, then read the
*Notice* line under it.** You do not write anything in this part.

By A5 the message that ended Module 2 works. By A7 you have a number for how
often the whole thing is right.


### Setup — run this first

Your Module 2 tools, rebuilt: `clean_text()` and `fix_typos()`. Nothing new,
**except four lines in the middle** — read the `KEEP` block.

Module 2 told you that filler words carry no information. That was true when you
were matching one keyword. It stops being true today: `where is my parcel` and
`when will my parcel arrive` are two different customer goals, and the only
difference between them is a filler word.

In [ ]:
import string
from difflib import get_close_matches

try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    NLTK_STOP = set(stopwords.words("english"))
    print("NLTK list loaded:", len(NLTK_STOP), "words")
except Exception:
    NLTK_STOP = set("i me my we you he she it they a an the is am are was were "
                    "be been do does did have has had of to in on at for with "
                    "and or but if this that these those not no can will just".split())
    print("Download blocked - using the built-in list:", len(NLTK_STOP), "words")

# The four question words are the whole difference between two intents today,
# and the standard list throws all of them away. So we take them back.
KEEP = {"where", "when", "how", "why", "what", "not", "no", "before", "after"}
STOP = NLTK_STOP - KEEP
print("After keeping the question words:", len(STOP), "words")

KEYWORDS = ["order", "parcel", "delivery", "refund", "return", "hello", "bye"]


def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    return [w for w in words if w not in STOP]


def fix_typos(words, cutoff=0.7):
    fixed = []
    for w in words:
        match = get_close_matches(w, KEYWORDS, n=1, cutoff=cutoff)
        fixed.append(match[0] if match else w)
    return fixed


print(clean_text("Where is my parcel?"))


**Notice:** `198` words became `189`. Nine words came back, and
`['where', 'parcel']` is the proof — Module 2's cleaner would have handed you
`['parcel']` and thrown the question away.

This is the first time in the course you have **changed** a tool because the job
changed. Module 2 was right; today's job is different.

### A1 — The message that beat you last module

Here is your Module 2 bot, with cleaning and typo-fixing in front of it. Run it
on the message that ended the last lesson.

In [ ]:
def bot(msg):
    if "hello" in msg:
        return "Hi! How can I help?"
    elif "order" in msg:
        return "Your order is on the way."
    elif "bye" in msg:
        return "Goodbye!"
    else:
        return "Sorry, I didn't catch that."


message = "Where is my parcel?"
cleaned = fix_typos(clean_text(message))
print(cleaned)
print(bot(" ".join(cleaned)))


**Notice:** `['where', 'parcel']` — cleaned perfectly, spelled perfectly, and
the bot still says **`Sorry, I didn't catch that.`**

The branch says `order`. The customer said `parcel`. One keyword, one chance, and
the chance was missed.

The fix is not a better keyword. It is to stop looking for **one** word.

### A2 — The word lists

An **intent** is the goal behind the words. *Where's my parcel*, *has my order
shipped* and *tracking pls* are three sentences with one goal.

So give each goal a list of words that tend to show up in it. Five intents, and a
sixth — `fallback` — that catches everything else.

In [ ]:
INTENTS = {
    "check_order_status": {"where", "order", "parcel", "package", "delivery",
                           "tracking", "status", "receive", "arrived"},
    "return_item":        {"return", "refund", "exchange", "back", "broken",
                           "wrong", "size", "shoes", "bag"},
    "delivery_time":      {"when", "how", "long", "time", "days", "arrive",
                           "deliver", "shipping", "fast"},
    "greeting":           {"hello", "hi", "hey", "morning", "afternoon",
                           "evening", "greetings"},
    "goodbye":            {"bye", "goodbye", "thanks", "thank", "tq", "ok",
                           "cheers"},
}

# The typo-fixer should aim at every word we now care about, not the old seven.
ALL_WORDS = set()
for word_list in INTENTS.values():
    ALL_WORDS = ALL_WORDS | word_list

KEYWORDS = sorted(ALL_WORDS)
print(len(INTENTS), "intents,", len(KEYWORDS), "words in total")
print(KEYWORDS)


**Notice:** there is no `fallback` list, and there never will be. `fallback` is
what happens when nothing else scores — it is the absence of a match, not a list
of words.

Also notice what just happened to `fix_typos`: it now aims at **41 words**
instead of 7. That makes it much better at rescuing `ordr`, and much more
dangerous. You see the damage in A7.

### A3 — Scoring by counting

This is the whole algorithm. For each intent, count how many of the message's
words appear in that intent's list.

That is it. No maths beyond `+ 1`.

In [ ]:
def score_intents(words):
    scores = {}
    for intent, word_list in INTENTS.items():
        count = 0
        for w in words:
            if w in word_list:
                count += 1
        scores[intent] = count
    return scores


words = fix_typos(clean_text("Where is my parcel?"))
print(words)
for intent, score in score_intents(words).items():
    print(f"  {intent:20} {score}")


**Notice:** `check_order_status` scores **2** and everything else scores **0**.

| Message word | check_order_status | return_item | delivery_time |
| --- | --- | --- | --- |
| `where` | ✅ | | |
| `parcel` | ✅ | | |
| **Score** | **2** | **0** | **0** |

You could have done that on paper, and that matters. Nothing in this module is
hidden from you — if the bot picks the wrong intent, you can always point at the
word that made it.

### A4 — Pick the winner

Highest score wins.

In [ ]:
scores = score_intents(fix_typos(clean_text("Where is my parcel?")))
best = max(scores, key=scores.get)

print(scores)
print("winner:", best, "with", scores[best])


**Notice:** `max(scores, key=scores.get)` is the one line here you have not seen
before. It means *give me the key with the biggest value*. That is all. It is not
what this module is teaching, so do not spend time on it.

There is something it does quietly that you will meet again in B4: if two
intents **tie**, it returns whichever one is written first in `INTENTS`, and it
does not tell you there was a tie.

### A5 — The not-sure rule

What if the winning score is 1? Or 0? Then the bot has one word's worth of
evidence and should not pretend otherwise.

So: **if the best score is below the cut-off, say `fallback` instead.**

In [ ]:
def classify(text, cutoff=2):
    words = fix_typos(clean_text(text))
    scores = score_intents(words)
    best = max(scores, key=scores.get)
    if scores[best] < cutoff:
        return "fallback", scores
    return best, scores


for m in ["Where is my parcel?", "i wan 2 chk my ordr", "how long to deliver to penang"]:
    intent, scores = classify(m)
    print(f"{m:32} -> {intent:20} {max(scores.values())}")


**Notice the first line. `Where is my parcel?` → `check_order_status`.**

That is the message that ended Module 1 *and* Module 2. It works now, and it
works for a boring reason: `parcel`, `package` and `delivery` are all sitting in
the `check_order_status` list next to `order`.

Now notice the second line. `i wan 2 chk my ordr` scores only **1** — `ordr` was
rescued to `order`, but nothing else in it is a word we know — so the bot says
`fallback` rather than guessing. Last module that message was a failure. Today it
is an *admitted* failure, which is a different and better thing.

### A6 — Turn the cut-off

`cutoff=2` was a guess. Try three of them on three messages.

In [ ]:
tests = ["wat time u all close on sunday", "where is my order", "hi"]

for c in [1, 2, 3]:
    print(f"--- cutoff = {c} ---")
    for m in tests:
        print(f"  {m:32} -> {classify(m, cutoff=c)[0]}")


**Notice: no cut-off gets all three right.** Say that out loud, because it looks
like you have made a mistake and you have not.

| | `wat time u all close on sunday` | `where is my order` | `hi` |
| --- | --- | --- | --- |
| **1** | `delivery_time` ❌ confidently wrong | `check_order_status` ✅ | `greeting` ✅ |
| **2** | `fallback` ✅ | `check_order_status` ✅ | `fallback` ❌ |
| **3** | `fallback` ✅ | `fallback` ❌ | `fallback` ❌ |

At `1` the bot answers a question about opening hours with a delivery estimate.
At `3` a greeting falls through, because `hi` is one word and one word can never
score 3.

The cut-off is not a setting with a correct value. It is a choice about which
kind of mistake you would rather make.

### A7 — Measure

Module 2 ended with a number on the board. Here is today's, on the 32 messages in
the course bank — and this time it is **accuracy**: not *did the bot reply* but
*did it pick the right intent*.

The bank has a labelled `intent` column. That is the right answer, written by a
human, and it is what we check against.

In [ ]:
BANK = [
    ("where is my order", "check_order_status"),
    ("hi, my order still havent arrive", "check_order_status"),
    ("ORD-48210 where??", "check_order_status"),
    ("parcel say delivered but i never receive", "check_order_status"),
    ("can u check my ordr status", "check_order_status"),
    ("still waiting for my delivery leh", "check_order_status"),
    ("my parcel not here yet", "check_order_status"),
    ("tracking pls", "check_order_status"),
    ("WHERE IS MY ORDER???", "check_order_status"),
    ("i wan 2 chk my ordr", "check_order_status"),
    ("can i return the shoes i bought last week ah", "return_item"),
    ("i want to return the blue shoes", "return_item"),
    ("how to refund", "return_item"),
    ("the bag is broken, i want exchange", "return_item"),
    ("wrong size how", "return_item"),
    ("i dont want it anymore can send back?", "return_item"),
    ("how long to deliver to penang", "delivery_time"),
    ("when will it arrive", "delivery_time"),
    ("how many days for shipping", "delivery_time"),
    ("can arrive before raya or not", "delivery_time"),
    ("delivery time to sabah?", "delivery_time"),
    ("Hello", "greeting"),
    ("hi", "greeting"),
    ("good morning", "greeting"),
    ("hey there", "greeting"),
    ("ok thanks bye", "goodbye"),
    ("tq bye", "goodbye"),
    ("This is the THIRD time I'm asking. Absolutely useless.", "complaint"),
    ("Wow, fantastic. Third delivery failure this month.", "complaint"),
    ("Great service as always. Package arrived smashed.", "complaint"),
    ("wat time u all close on sunday", "fallback"),
    ("why so expensive one", "fallback"),
]


def measure(cutoff=2):
    correct = 0
    not_sure = 0
    for message, true_intent in BANK:
        guess = classify(message, cutoff=cutoff)[0]
        if guess == true_intent:
            correct += 1
        if guess == "fallback":
            not_sure += 1
    return correct, not_sure


correct, not_sure = measure(cutoff=2)
print(f"Correct:   {correct} / {len(BANK)}")
print(f"Not sure:  {not_sure} / {len(BANK)}")
print()
print("Getting it wrong:")
for message, true_intent in BANK:
    guess = classify(message, cutoff=2)[0]
    if guess != true_intent:
        print(f"  {message[:44]:44} want {true_intent:19} got {guess}")


**Notice: 15 out of 32.** That is worse than it should be, and the second number
says why — **18 of the 32 got `fallback`.** The bot is not picking wrong
intents. It is refusing to pick at all.

Three other things in that list are worth a look:

- `Hello`, `hi`, `hey there` and `good morning` all fail. A greeting is one word.
  It can never score 2.
- Three messages are labelled `complaint` and **we do not have a `complaint`
  intent**, so they cannot possibly be right. That is task B3.
- `Great service as always. Package arrived smashed.` scores
  `check_order_status` **2** and is called confidently. It is sarcasm, it is a
  complaint, and counting words will never see that. Module 7.

The cut-off is the thing to fix first. That is task B2.

---
# Part B — Your turn

Six tasks. You are expected to get things wrong here — that is what the part is
for. Nothing in Part B is marked.

If a cell will not run, read the **last** line of the red error first. It is
usually the useful one.

### B1 — Predict, then run

**Fill in the prediction column before you run anything.** Double-click the table
to type in it.

You have the word lists from A2 in front of you. Count by hand, the way you did
on the whiteboard — then find out whether you and the computer agree.

| Message | Words that survive cleaning | Winning intent | Its score | Right or wrong? |
| --- | --- | --- | --- | --- |
| `tracking pls` | | | | |
| `how to refund` | | | | |
| `ok thanks bye` | | | | |
| `good morning` | | | | |

Now run it.

In [ ]:
tests = ["tracking pls", "how to refund", "ok thanks bye", "good morning"]

for m in tests:
    words = fix_typos(clean_text(m))
    scores = score_intents(words)
    best = max(scores, key=scores.get)
    print(f"{m:16} {str(words):34} -> {best:20} {scores[best]}")
    print(f"{'':16} {scores}")


**Which one surprised you, and why?**

_______________________________________________

_______________________________________________

### B2 — Turn the cut-off dial

A7 measured **15/32** with `cutoff=2`, and **18** messages came back `fallback`.

Run the same measurement at every cut-off from 0 to 4 and find out what the
number is actually costing you.

**Fill in the blanks first.** `measure()` is already written — you are calling
it, not writing it.

In [ ]:
for c in [___, ___, ___, ___, ___]:
    correct, not_sure = measure(cutoff=___)
    print(f"cutoff={c}   correct {correct}/32   said not sure {not_sure}/32")


**Fill this in from what you got:**

| Cut-off | Correct | Said "not sure" | What is wrong with it |
| --- | --- | --- | --- |
| `0` | | | |
| `1` | | | |
| `2` | | | |
| `3` | | | |
| `4` | | | |

Now the question that matters. `cutoff=1` scores highest. Run this before you
decide it is the answer:

```python
print(classify("wat time u all close on sunday", cutoff=1))
print(classify("why so expensive one", cutoff=0))
```

**Which cut-off would you ship, and what are you accepting when you choose it?**

_______________________________________________

_______________________________________________

### B3 — Write your own word list

Three messages in the bank are labelled `complaint` and your bot **cannot ever
get them right**, because there is no `complaint` intent. Not a bug — a missing
intent.

So write one. The three messages are:

```
This is the THIRD time I'm asking. Absolutely useless.
Wow, fantastic. Third delivery failure this month.
Great service as always. Package arrived smashed.
```

**Rules, and they are the whole task:**

1. Between 8 and 12 words.
2. Every word must be one a customer would actually type.
3. **No word may already appear in another intent's list.** Check before you add.

In [ ]:
INTENTS["complaint"] = {
    "useless", "___", "___", "___",
    "___", "___", "___", "___",
}

ALL_WORDS = set()
for word_list in INTENTS.values():
    ALL_WORDS = ALL_WORDS | word_list
KEYWORDS = sorted(ALL_WORDS)

correct, not_sure = measure(cutoff=___)
print(f"With your complaint intent: {correct}/32")


**Now break it on purpose.** A classmate says the list is too thin and more
words must be better. Add these four and measure again:

`parcel`, `broken`, `delivery`, `not`

Every one of them is a word a complaining customer really does type. Every one of
them is already in another intent's list.

In [ ]:
if "complaint" not in INTENTS:
    raise SystemExit(
        "Go back and finish B3 first — this cell builds on the complaint "
        "word list you wrote there."
    )

INTENTS["complaint"] = INTENTS["complaint"] | {"parcel", "broken", "delivery", "not"}

correct, not_sure = measure(cutoff=1)
print(f"After adding four more words: {correct}/32")
print()
print(classify("my parcel not here yet", cutoff=1))


**The score went down. Which message did you lose, and why?**

_______________________________________________

_______________________________________________

Now take the four words back out — the next task needs a working bot.

In [ ]:
if "complaint" not in INTENTS:
    raise SystemExit(
        "Go back and finish B3 first — this cell builds on the complaint "
        "word list you wrote there."
    )

INTENTS["complaint"] = INTENTS["complaint"] - {"parcel", "broken", "delivery", "not"}

correct, not_sure = measure(cutoff=1)
print(f"Back to: {correct}/32")


### B4 — Find the bug

A classmate says `order` obviously belongs in `delivery_time` as well, because
*"how long will my order take"* is a delivery question. It sounds right.

Their version is below. **Run it first, then read it and work out what happened
— do not fix it by guessing.**

In [ ]:
INTENTS_BUGGY = {
    "check_order_status": {"where", "order", "parcel", "package", "delivery",
                           "tracking", "status", "receive", "arrived"},
    "return_item":        {"return", "refund", "exchange", "back", "broken",
                           "wrong", "size", "shoes", "bag"},
    "delivery_time":      {"when", "how", "long", "time", "days", "arrive",
                           "deliver", "shipping", "fast", "order"},
    "greeting":           {"hello", "hi", "hey", "morning", "afternoon",
                           "evening", "greetings"},
    "goodbye":            {"bye", "goodbye", "thanks", "thank", "tq", "ok",
                           "cheers"},
}

GOOD = INTENTS
message = "hi, my order still havent arrive"

INTENTS = INTENTS_BUGGY
print("buggy: ", classify(message, cutoff=1))
INTENTS = GOOD
print("normal:", classify(message, cutoff=1))


**Write your answer here.** The customer is asking where their order is. What
does the buggy bot reply about, and which single word caused it?

_______________________________________________

<br><br><br><br><br><br>

**The answer:** `order` is now in **two** word lists. The message contains
`order` and `arrive`, so `delivery_time` scores 2 and `check_order_status` scores
1 — and the customer chasing a late parcel is told the standard delivery time
instead.

Nothing errored. Nothing warned. The reply is polite, fluent and about the wrong
thing.

### B5 — Your own ten messages

Since Module 1 you have been collecting real customer messages. Module 2 scored
them on *did the bot reply at all*. Today the question is harder: **did it pick
the right intent?**

Paste your ten in, and **label each one yourself first** — that label is the
right answer, and you are the only one who can write it.

In [ ]:
# ("the message", "the intent you say it really is")
my_messages = [
    ("where is my parcel", "check_order_status"),
    ("can i change the size", "return_item"),
    ("___", "___"),
    ("___", "___"),
    ("___", "___"),
    # ... all 10
]

my_messages = [(m, i) for m, i in my_messages if m != "___"]

CUTOFF = 1        # <-- the one you chose in B2

correct = 0
for message, true_intent in my_messages:
    guess = classify(message, cutoff=CUTOFF)[0]
    mark = "OK " if guess == true_intent else "NO "
    if guess == true_intent:
        correct += 1
    print(f"{mark} {message[:40]:40} you: {true_intent:20} bot: {guess}")

print()
print(f"Right intent: {correct} / {len(my_messages)}")


**Pick one the bot got wrong. Which word list would you change, and what would
that change break?**

_______________________________________________

_______________________________________________

### B6 — The one it still cannot do

Last task. Nothing to fix here — the point is to find the edge.

Run it.

In [ ]:
pair = [
    "where is my order",
    "where is my order ORD-48210",
]

for m in pair:
    intent, scores = classify(m, cutoff=1)
    print(f"{m:32} -> {intent:20} score {max(scores.values())}")
    print(f"{'':32}    {fix_typos(clean_text(m))}")


**Both messages get the same intent and the same score.** One of them tells you
exactly which order. The other does not.

**What extra thing does the bot need before it can actually help the first
customer — and could any word list ever hold it?**

_______________________________________________

_______________________________________________

---
## Stretch — let the computer write the word lists *(only if you finish early)*

**Optional. Never assessed, and Module 4 does not assume it.**

Writing word lists by hand never ends. A real system learns them from labelled
examples instead. Below are four lines of scikit-learn that do exactly that: it
reads the 32 labelled messages and works out the useful words itself.

Run it, then read the two numbers very carefully.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

X = [m for m, i in BANK]
y = [i for m, i in BANK]

# --- 1. train on all 32, then test on the same 32 ---
v = CountVectorizer()
model = MultinomialNB().fit(v.fit_transform(X), y)
seen = sum(p == t for p, t in zip(model.predict(v.transform(X)), y))
print(f"Tested on the messages it learned from: {seen} / 32")

# --- 2. hide one message, learn from the other 31, guess the hidden one ---
hidden_correct = 0
for i in range(len(X)):
    v2 = CountVectorizer()
    m2 = MultinomialNB().fit(v2.fit_transform(X[:i] + X[i+1:]), y[:i] + y[i+1:])
    hidden_correct += m2.predict(v2.transform([X[i]]))[0] == y[i]
print(f"Tested on messages it had never seen:   {hidden_correct} / 32")


**Two questions.**

**1.** The first number is far better than your hand-built scorer. The second is
far worse. Which one is the honest measure of how good this thing is?

_______________________________________________

**2.** Your word lists can be read. You can point at `parcel` and say *that is
why it chose order status*. What can you point at inside the scikit-learn model?

_______________________________________________

<br><br><br><br><br><br>

**Answers.**

**1.** The second. The first number, **31/32**, is the model being asked
questions it has already been given the answers to — it memorised them. The
honest number is **14/32**, and it is worse than your counting bot's 27. With 32
examples there is not enough to learn from; with 32,000 it would win easily, and
that is the real trade: hand-written lists work on day one, learned lists need
data you do not have yet.

**2.** Nothing you can read. The model is a table of numbers. When it gets a
customer's message wrong you cannot point at the word that did it, you can only
retrain and hope. Your 41-word version is worse and **explainable**; that is a
genuine engineering choice and plenty of real systems make it.


---
## Homework

**Add five words to each intent's word list.** Then test your bot on ten messages
you write yourself — new ones, not the ones from class.

Record the score **before** and **after** your five words.

> **Some of you will make it worse.** That is the interesting result, not the
> embarrassing one. If your score dropped, find the message you lost and the word
> that lost it, and bring both to the next lesson.

| | Score |
| --- | --- |
| Before your extra words | ______ / 10 |
| After your extra words | ______ / 10 |

**If it went down:** which word, and which intent stole which message?

_______________________________________________